# Chat Template Formatting for Instruction Tuning

> Understand and implement chat templates for different LLM families

**Why Chat Templates Matter:**
- Different models expect different conversation formats
- Proper formatting = better instruction following
- Incorrect formatting = degraded performance

**Models Covered:**
- LLaMA 2/3 Chat
- Mistral Instruct
- ChatML (OpenAI, Qwen)
- Vicuna
- Zephyr

In [ ]:
import json
from typing import List, Dict, Optional
from dataclasses import dataclass
from datetime import datetime

## 1. Chat Message Data Structure

In [ ]:
@dataclass
class ChatMessage:
    """Standard chat message format"""
    role: str  # 'system', 'user', 'assistant'
    content: str
    metadata: Optional[Dict] = None

# Example conversation
conversation = [
    ChatMessage(
        role='system',
        content='You are a helpful AI assistant. Answer questions accurately and concisely.'
    ),
    ChatMessage(
        role='user',
        content='What is machine learning?'
    ),
    ChatMessage(
        role='assistant',
        content='Machine learning is a subset of AI where systems learn patterns from data without explicit programming.'
    ),
    ChatMessage(
        role='user',
        content='Give me an example.'
    ),
    ChatMessage(
        role='assistant',
        content='Email spam filters use ML to learn which emails are spam based on features like sender, subject, and content.'
    ),
]

print(f"Conversation has {len(conversation)} messages")
for msg in conversation:
    print(f"\n[{msg.role.upper()}]")
    print(f"  {msg.content[:80]}...")

## 2. LLaMA 2/3 Chat Template

In [ ]:
class LlamaChatTemplate:
    """
    LLaMA 2/3 Chat template format.
    
    Format:
    <s>[INST] <<SYS>>\n    {system_prompt}\n    <</SYS>>\n    \n    {user_message} [/INST] {assistant_response} </s>
    <s>[INST] {user_message} [/INST]
    """
    
    BOS = '<s>'
    EOS = '</s>'
    INST_START = '[INST]'
    INST_END = '[/INST]'
    SYS_START = '<<SYS>>'
    SYS_END = '<</SYS>>'
    
    def apply(self, messages: List[ChatMessage]) -> str:
        """Apply LLaMA chat template to messages"""
        if not messages:
            return ""
        
        formatted = []
        system_msg = None
        
        # Extract system message
        if messages[0].role == 'system':
            system_msg = messages[0].content
            messages = messages[1:]
        
        # Process conversation pairs
        i = 0
        while i < len(messages):
            if messages[i].role == 'user':
                user_content = messages[i].content
                
                # Build instruction block
                if i == 0 and system_msg:
                    # First turn with system prompt
                    inst = f"{self.INST_START} {self.SYS_START}\n{system_msg}\n{self.SYS_END}\n\n{user_content} {self.INST_END}"
                else:
                    inst = f"{self.INST_START} {user_content} {self.INST_END}"
                
                # Add assistant response if present
                if i + 1 < len(messages) and messages[i + 1].role == 'assistant':
                    assistant_content = messages[i + 1].content
                    formatted.append(f"{self.BOS}{inst} {assistant_content}{self.EOS}")
                    i += 2
                else:
                    # Last turn (for generation)
                    formatted.append(f"{self.BOS}{inst}")
                    i += 1
            else:
                i += 1
        
        return '\n'.join(formatted)
    
    def format_for_training(self, messages: List[ChatMessage]) -> Dict:
        """Format for SFT training with loss masking"""
        prompt = self.apply(messages[:-1] if messages[-1].role == 'assistant' else messages)
        
        # For training, we mask user tokens and only compute loss on assistant tokens
        full_text = self.apply(messages)
        
        return {
            'text': full_text,
            'prompt': prompt,
            'response': messages[-1].content if messages[-1].role == 'assistant' else '',
        }

# Test LLaMA template
llama_template = LlamaChatTemplate()
llama_formatted = llama_template.apply(conversation)

print("🦙 LLaMA 2/3 Chat Template:")
print("=" * 60)
print(llama_formatted)
print("=" * 60)

## 3. ChatML Template (OpenAI, Qwen, Mistral)

In [ ]:
class ChatMLTemplate:
    """
    ChatML format used by OpenAI, Qwen, and some Mistral models.
    
    Format:
    <|im_start|>system\n    {system_message}<|im_end|>\n    <|im_start|>user\n    {user_message}<|im_end|>\n    <|im_start|>assistant\n    {assistant_message}<|im_end|>
    """
    
    IM_START = '<|im_start|>'
    IM_END = '<|im_end|>'
    
    def apply(self, messages: List[ChatMessage]) -> str:
        """Apply ChatML template"""
        formatted = []
        
        for msg in messages:
            formatted.append(f"{self.IM_START}{msg.role}\n{msg.content}{self.IM_END}")
        
        # Add assistant start for generation
        if messages and messages[-1].role == 'user':
            formatted.append(f"{self.IM_START}assistant\n")
        
        return '\n'.join(formatted)
    
    def format_for_training(self, messages: List[ChatMessage]) -> Dict:
        """Format for training with proper masking"""
        # Build full conversation
        text = self.apply(messages)
        
        # Build prompt (everything except last assistant response)
        prompt_messages = messages[:-1] if messages[-1].role == 'assistant' else messages
        prompt = self.apply(prompt_messages)
        
        return {
            'text': text,
            'prompt': prompt,
            'response': messages[-1].content if messages[-1].role == 'assistant' else '',
        }

# Test ChatML
chatml_template = ChatMLTemplate()
chatml_formatted = chatml_template.apply(conversation)

print("💬 ChatML Template:")
print("=" * 60)
print(chatml_formatted)
print("=" * 60)

## 4. Mistral Instruct Template

In [ ]:
class MistralTemplate:
    """
    Mistral Instruct template.
    
    Format:
    <s>[INST] {user_message} [/INST] {assistant_response}</s>
    [INST] {user_message} [/INST]
    
    Note: No system prompt in standard Mistral.
    """
    
    BOS = '<s>'
    EOS = '</s>'
    INST_START = '[INST]'
    INST_END = '[/INST]'
    
    def apply(self, messages: List[ChatMessage]) -> str:
        """Apply Mistral template"""
        formatted = []
        system_msg = None
        
        # Extract system (prepend to first user message)
        start_idx = 0
        if messages and messages[0].role == 'system':
            system_msg = messages[0].content
            start_idx = 1
        
        i = start_idx
        first_turn = True
        
        while i < len(messages):
            if messages[i].role == 'user':
                user_content = messages[i].content
                
                # Prepend system to first user message
                if first_turn and system_msg:
                    user_content = f"{system_msg}\n\n{user_content}"
                
                inst = f"{self.INST_START} {user_content} {self.INST_END}"
                
                if i + 1 < len(messages) and messages[i + 1].role == 'assistant':
                    assistant_content = messages[i + 1].content
                    prefix = self.BOS if first_turn else ''
                    formatted.append(f"{prefix}{inst} {assistant_content}{self.EOS}")
                    i += 2
                else:
                    prefix = self.BOS if first_turn else ''
                    formatted.append(f"{prefix}{inst}")
                    i += 1
                
                first_turn = False
            else:
                i += 1
        
        return '\n'.join(formatted)

# Test Mistral
mistral_template = MistralTemplate()
mistral_formatted = mistral_template.apply(conversation)

print("🌪️ Mistral Instruct Template:")
print("=" * 60)
print(mistral_formatted)
print("=" * 60)

## 5. Zephyr Template

In [ ]:
class ZephyrTemplate:
    """
    Zephyr (Hugging Face) chat template.
    
    Format:
    <|system|>\n    {system_message}</s>\n    <|user|>\n    {user_message}</s>\n    <|assistant|>\n    {assistant_message}
    """
    
    def apply(self, messages: List[ChatMessage]) -> str:
        """Apply Zephyr template"""
        formatted = []
        
        for msg in messages:
            if msg.role == 'system':
                formatted.append(f"<|system|>\n{msg.content}</s>")
            elif msg.role == 'user':
                formatted.append(f"<|user|>\n{msg.content}</s>")
            elif msg.role == 'assistant':
                formatted.append(f"<|assistant|>\n{msg.content}")
        
        # Add assistant marker for generation
        if messages and messages[-1].role == 'user':
            formatted.append("<|assistant|>\n")
        
        return '\n'.join(formatted)

# Test Zephyr
zephyr_template = ZephyrTemplate()
zephyr_formatted = zephyr_template.apply(conversation)

print("🌬️ Zephyr Template:")
print("=" * 60)
print(zephyr_formatted)
print("=" * 60)

## 6. Side-by-Side Comparison

In [ ]:
# Compare all templates
templates = {
    'LLaMA 2/3': llama_template,
    'ChatML': chatml_template,
    'Mistral': mistral_template,
    'Zephyr': zephyr_template,
}

print("📊 Template Comparison (First Turn Only):")
print("=" * 80)

first_turn = conversation[:3]  # system + first user + first assistant

for name, template in templates.items():
    formatted = template.apply(first_turn)
    token_count = len(formatted.split())  # Rough word count
    print(f"\n🔹 {name} ({token_count} tokens approx):")
    print("-" * 40)
    print(formatted[:300] + ('...' if len(formatted) > 300 else ''))
    print()

## 7. Training Data Formatting

In [ ]:
# Create training dataset with different templates
training_examples = [
    {
        'messages': [
            {'role': 'system', 'content': 'You are a helpful coding assistant.'},
            {'role': 'user', 'content': 'Write a Python function to reverse a string.'},
            {'role': 'assistant', 'content': '```python\ndef reverse_string(s):\n    return s[::-1]\n```'},
        ]
    },
    {
        'messages': [
            {'role': 'system', 'content': 'You are a math tutor.'},
            {'role': 'user', 'content': 'What is the derivative of x^2?'},
            {'role': 'assistant', 'content': 'The derivative of x² with respect to x is 2x, using the power rule: d/dx(x^n) = n·x^(n-1).'},
        ]
    },
]

def convert_to_chatmessages(example: Dict) -> List[ChatMessage]:
    """Convert dict format to ChatMessage objects"""
    return [ChatMessage(role=m['role'], content=m['content']) for m in example['messages']]

# Format for LLaMA training
print("📚 Training Dataset (LLaMA format):")
print("=" * 60)

for i, example in enumerate(training_examples):
    messages = convert_to_chatmessages(example)
    train_format = llama_template.format_for_training(messages)
    
    print(f"\n--- Example {i+1} ---")
    print(f"Prompt length: {len(train_format['prompt'])} chars")
    print(f"Response length: {len(train_format['response'])} chars")
    print(f"Total length: {len(train_format['text'])} chars")
    print(f"\nPrompt:\n{train_format['prompt'][:150]}...")
    print(f"\nResponse:\n{train_format['response'][:100]}...")

## 8. Loss Masking for Training

In [ ]:
def create_loss_mask(text: str, prompt: str, tokenizer=None) -> List[int]:
    """
    Create loss mask where:
    - 0 = mask (don't compute loss) for prompt tokens
    - 1 = compute loss for response tokens
    """
    # Simple character-level masking (in practice, use tokenizer)
    prompt_len = len(prompt)
    total_len = len(text)
    
    mask = [0] * prompt_len + [1] * (total_len - prompt_len)
    
    return mask

# Example
messages = convert_to_chatmessages(training_examples[0])
train_data = llama_template.format_for_training(messages)

mask = create_loss_mask(train_data['text'], train_data['prompt'])

print("🎭 Loss Masking Example:")
print("=" * 60)
print(f"Total tokens: {len(mask)}")
print(f"Masked (prompt): {mask.count(0)} tokens")
print(f"Trainable (response): {mask.count(1)} tokens")
print(f"\nMask pattern: {mask[:50]}...")
print(f"(0 = masked, 1 = train)")

# Visualize
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 2))
ax.imshow([mask], aspect='auto', cmap='RdYlGn', interpolation='nearest')
ax.set_xlabel('Token Position')
ax.set_yticks([])
ax.set_title('Loss Mask: Green = Train, Red = Masked')
plt.tight_layout()
plt.show()

## 9. Multi-Turn Conversation Handling

In [ ]:
# Complex multi-turn conversation
multi_turn = [
    ChatMessage(role='system', content='You are a travel assistant.'),
    ChatMessage(role='user', content='I want to visit Japan.'),
    ChatMessage(role='assistant', content='Japan is amazing! When are you planning to visit?'),
    ChatMessage(role='user', content='Next spring, during cherry blossom season.'),
    ChatMessage(role='assistant', content='Perfect timing! Late March to early April is peak sakura season.'),
    ChatMessage(role='user', content='What cities should I visit?'),
    ChatMessage(role='assistant', content='Tokyo, Kyoto, and Osaka are must-sees.'),
]

print("🗾 Multi-Turn Conversation (LLaMA format):")
print("=" * 60)
multi_formatted = llama_template.apply(multi_turn)
print(multi_formatted)
print("=" * 60)

# Count turns
user_turns = sum(1 for m in multi_turn if m.role == 'user')
assistant_turns = sum(1 for m in multi_turn if m.role == 'assistant')
print(f"\nTotal turns: {user_turns} user, {assistant_turns} assistant")
print(f"Formatted length: {len(multi_formatted)} characters")

## 🎯 Exercises

1. **Custom Template**: Create a template for a new model family (e.g., Gemma, Phi).
2. **Tool Use**: Extend templates to support function calling / tool use format.
3. **JSON Mode**: Implement JSON mode formatting for structured output.
4. **Batch Processing**: Write a function to format an entire dataset for training.
5. **Hugging Face**: Use `transformers.AutoTokenizer.apply_chat_template()` and compare.
6. **Conversation Truncation**: Implement sliding window for long conversations.